# Generating data from long range arena

## 1. Installing python packages

In [ ]:
!pip install uv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 103.6 MB/s eta 0:00:00


### 1.1 Installing all libraries for the code to run

In [ ]:
!uv pip install datasets huggingface_hub lightning numpy pandas pyarrow torch tqdm wandb optuna optuna-integration[pytorch_lightning]

Using Python 3.12.12 environment at: /usr
Resolved 84 packages in 536ms
Prepared 7 packages in 90ms
Installed 7 packages in 9ms
 + colorlog==6.10.1
 + lightning==2.6.1
 + lightning-utilities==0.15.2
 + optuna==4.7.0
 + optuna-integration==4.7.0
 + pytorch-lightning==2.6.1
 + torchmetrics==1.8.2


## 2. Importing Python libraries

In [ ]:
import pickle
import random
import sys
import re
import optuna
from concurrent.futures import ThreadPoolExecutor, as_completed
from functools import partial
import threading

from huggingface_hub import HfApi, login, hf_hub_download

import lightning as pl
import numpy as np
import torch
from datasets import load_dataset
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

## 3. Importing data from hugging face

if sucess, you need to run:"4.1 Declaring all PyTorch dataset classes to train ML models" to create the PyTorch dataset classes so that the baselines and models developed can use the data to train the models

In [ ]:
repo_id = "monteirot/lra-benchmarks"

files = ['listops.pt', 'imdb_lra.pt', 'acl_retrieval_lra.pt',
         'cifar10_sequential_lra.pt', 'pathfinder_lra.pt', 'pathx_lra.pt']

for f in files:
    print(f"Downloading {f}...")
    hf_hub_download(repo_id=repo_id, filename=f, repo_type="dataset", local_dir=".")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


listops.pt:   0%|          | 0.00/175M [00:00<?, ?B/s]

imdb_lra.pt:   0%|          | 0.00/132M [00:00<?, ?B/s]

acl_retrieval_lra.pt:   0%|          | 0.00/274M [00:00<?, ?B/s]

cifar10_sequential_lra.pt:   0%|          | 0.00/124M [00:00<?, ?B/s]

pathfinder_lra.pt:   0%|          | 0.00/1.64G [00:00<?, ?B/s]

pathx_lra.pt:   0%|          | 0.00/26.2G [00:00<?, ?B/s]

## 4. Long Range Arena benchmark

The deep learning model requires independent hyperparameter tuning runs across all six benchmarks.

Testing one optimal model against four optimal baselines results in 30 total runs (6 runs × 5 models).

Each run includes hyperparameter tuning and metric-based verification.

Benchmarks are implemented from scratch based on Long Range Arena: A Benchmark for Efficient Transformers.

Link: https://arxiv.org/abs/2011.04006

### 4.1 Declaring all PyTorch dataset classes to train ML models

In [ ]:
class ListOpsDataset(Dataset):
    def __init__(self, data, max_len=2048):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Pad or truncate
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
class IMDbDataset(Dataset):
    """PyTorch Dataset for byte-level IMDb classification."""
    def __init__(self, data, max_len=4096):
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Truncate or pad to max_len
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
class ACLRetrievalDataset(Dataset):
    """PyTorch Dataset for ACL document retrieval."""
    def __init__(self, data, max_len=8001):  # 4000 + 1 (sep) + 4000
        self.data = data
        self.max_len = max_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Truncate or pad
        if len(tokens) < self.max_len:
            mask = [1] * len(tokens) + [0] * (self.max_len - len(tokens))
            tokens = tokens + [0] * (self.max_len - len(tokens))
        else:
            tokens = tokens[:self.max_len]
            mask = [1] * self.max_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
class CIFAR10SequentialDataset(Dataset):
    """PyTorch Dataset for sequential CIFAR-10 classification."""
    def __init__(self, data, seq_len=1024):
        self.data = data
        self.seq_len = seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens, label = self.data[idx]

        # Pad or truncate to seq_len
        if len(tokens) < self.seq_len:
            mask = [1] * len(tokens) + [0] * (self.seq_len - len(tokens))
            tokens = tokens + [0] * (self.seq_len - len(tokens))
        else:
            tokens = tokens[:self.seq_len]
            mask = [1] * self.seq_len

        return {
            'input_ids': torch.tensor(tokens, dtype=torch.long),
            'attention_mask': torch.tensor(mask, dtype=torch.long),
            'labels': torch.tensor(label, dtype=torch.long)
        }


In [ ]:
class PathfinderDataset(Dataset):
    """
    PyTorch Dataset for Pathfinder/Path-X tasks.
    Now accepts tensors directly (no list conversion needed).
    """
    def __init__(self, tokens, labels, seq_len):
        # Accept either tensor tuple or legacy list format
        if isinstance(tokens, torch.Tensor):
            self.tokens = tokens
            self.labels = labels
            self.is_tensor = True
        else:
            # Legacy list format fallback
            self.data = tokens  # Actually list of (tokens, label) tuples
            self.is_tensor = False
        self.seq_len = seq_len

    def __len__(self):
        return len(self.tokens) if self.is_tensor else len(self.data)

    def __getitem__(self, idx):
        if self.is_tensor:
            tokens = self.tokens[idx]
            label = self.labels[idx]
        else:
            tokens, label = self.data[idx]
            tokens = torch.tensor(tokens, dtype=torch.long)
            label = torch.tensor(label, dtype=torch.long)

        # Tokens should already be correct length, but handle edge cases
        if len(tokens) < self.seq_len:
            mask = torch.cat([torch.ones(len(tokens)), torch.zeros(self.seq_len - len(tokens))])
            tokens = torch.cat([tokens, torch.zeros(self.seq_len - len(tokens), dtype=torch.long)])
        else:
            tokens = tokens[:self.seq_len]
            mask = torch.ones(self.seq_len)

        return {
            'input_ids': tokens.long(),
            'attention_mask': mask.long(),
            'labels': label.long()
        }

### 4.2 ListOps: Hierarchical parsing of mathematical expressions

 Can the model understand nested structure, like matching parentheses and applying operations in the right order?

In [ ]:
class ListOpsGenerator:
    def __init__(self, max_depth=10, max_args=10, seed=42, device=None):
        random.seed(seed)
        np.random.seed(seed)
        self.max_depth = max_depth
        self.max_args = max_args
        self.ops = ['MAX', 'MIN', 'MED', 'SM']
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        self.vocab = {'[PAD]': 0, '[': 1, ']': 2, 'MAX': 3, 'MIN': 4,
                      'MED': 5, 'SM': 6}
        for i in range(10):
            self.vocab[str(i)] = 7 + i
        self.vocab_size = len(self.vocab)
        self._token_pattern = re.compile(r'\[|\]|MAX|MIN|MED|SM|\d')

    def _eval(self, expr):
        expr = expr.strip()
        if expr.isdigit():
            return int(expr)
        expr = expr[1:-1].strip()
        parts = expr.split(None, 1)
        op = parts[0]
        if len(parts) == 1:
            return 0

        args, current, depth = [], [], 0
        for c in parts[1]:
            if c == '[':
                depth += 1
                current.append(c)
            elif c == ']':
                depth -= 1
                current.append(c)
            elif c == ' ' and depth == 0:
                if current:
                    args.append(''.join(current))
                    current = []
            else:
                current.append(c)
        if current:
            args.append(''.join(current))

        vals = [self._eval(arg) for arg in args]
        if not vals:
            return 0
        if op == 'MAX':
            return max(vals)
        elif op == 'MIN':
            return min(vals)
        elif op == 'MED':
            s = sorted(vals)
            n = len(s)
            return s[n//2] if n % 2 else (s[n//2-1] + s[n//2]) // 2
        else:
            return sum(vals) % 10

    def _gen_expr(self, depth=0, curr_len=0, target_len=1000):
        if curr_len > target_len * 1.5:
            term_prob = 0.8
        elif curr_len > target_len:
            term_prob = 0.5
        elif curr_len < target_len * 0.3 and depth < self.max_depth - 2:
            term_prob = 0.1
        else:
            term_prob = 0.25

        if depth >= self.max_depth or (depth > 0 and random.random() < term_prob):
            return str(random.randint(0, 9))

        op = random.choice(self.ops)
        num_args = random.randint(3, self.max_args) if curr_len < target_len * 0.5 else random.randint(2, 5)
        args = []
        for _ in range(num_args):
            arg = self._gen_expr(depth + 1, curr_len + len(args) * 10, target_len)
            args.append(arg)
            curr_len += len(arg)
        return f"[ {op} {' '.join(args)} ]"

    def tokenize(self, expr):
        return [self.vocab[t] for t in self._token_pattern.findall(expr)]

    def _generate_single(self, target_len, min_len, max_len):
        for _ in range(100):
            expr = self._gen_expr(target_len=target_len)
            if min_len <= len(expr) <= max_len:
                try:
                    return (self.tokenize(expr), self._eval(expr))
                except:
                    pass
        return None

    def generate(self, n, min_len=500, max_len=2000, num_workers=8):
        data, target_len = [], (min_len + max_len) // 2
        pbar = tqdm(total=n, desc="Generating samples")

        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = {executor.submit(self._generate_single, target_len, min_len, max_len): i
                       for i in range(min(n * 2, 1000))}
            while len(data) < n:
                for future in as_completed(futures):
                    if len(data) >= n:
                        break
                    result = future.result()
                    if result:
                        data.append(result)
                        pbar.update(1)
                    del futures[future]
                needed = n - len(data) - len(futures)
                if needed > 0:
                    for _ in range(min(needed * 2, 100)):
                        futures[executor.submit(self._generate_single, target_len, min_len, max_len)] = len(futures)
        pbar.close()
        return data[:n]

In [ ]:
gen = ListOpsGenerator(seed=42)

print("Generating ListOps dataset...\n")
print("Training set (96000 samples):")
train_data = gen.generate(96000)

print("\nValidation set (2000 samples):")
val_data = gen.generate(2000)

print("\nTest set (2000 samples):")
test_data = gen.generate(2000)

# Create datasets
train_ds = ListOpsDataset(train_data)
val_ds = ListOpsDataset(val_data)
test_ds = ListOpsDataset(test_data)

# Save
torch.save({'train': train_ds, 'val': val_ds, 'test': test_ds,
            'vocab_size': gen.vocab_size}, 'listops.pt')

print(f"\nSaved to listops.pt")
print(f"Vocab size: {gen.vocab_size}")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Generating ListOps dataset...

Training set (96000 samples):



Generating samples:   0%|          | 6/96000 [00:06<29:08:23,  1.09s/it]


KeyboardInterrupt: 

### 4.3 Text: Sentiment classification on IMDb reviews

Can the model figure out sentiment when reading raw characters instead of words across a long document?

In [ ]:
class IMDbGenerator:
    """
    IMDb byte-level text classification for Long Range Arena benchmark.
    - Binary classification (positive/negative sentiment)
    - Character/byte-level tokenization (vocab size 256)
    - Max sequence length: 4096 (as per LRA spec)
    """
    def __init__(self, max_len=4096, seed=42, device=None):
        random.seed(seed)
        np.random.seed(seed)
        self.max_len = max_len
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        # Byte-level vocabulary (256 possible byte values + PAD)
        self.vocab_size = 257  # 0 = PAD, 1-256 = byte values
        self.pad_token = 0

    def tokenize(self, text):
        """Convert text to byte-level tokens (1-256, 0 reserved for PAD)."""
        # Encode to bytes and shift by 1 (so 0 is reserved for padding)
        return [b + 1 for b in text.encode('utf-8', errors='replace')]

    def _process_single(self, example):
        """Process a single IMDb example."""
        text = example['text']
        label = example['label']  # 0 = negative, 1 = positive
        tokens = self.tokenize(text)
        return (tokens, label)

    def load_and_process(self, num_workers=8):
        """Load IMDb dataset and process to byte-level tokens."""
        print("Loading IMDb dataset from HuggingFace...")
        dataset = load_dataset('imdb')

        train_data = []
        test_data = []

        # Process training set
        print("\nProcessing training set (25000 samples):")
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = [executor.submit(self._process_single, ex) for ex in dataset['train']]
            for future in tqdm(as_completed(futures), total=len(futures), desc="Train"):
                train_data.append(future.result())

        # Process test set
        print("\nProcessing test set (25000 samples):")
        with ThreadPoolExecutor(max_workers=num_workers) as executor:
            futures = [executor.submit(self._process_single, ex) for ex in dataset['test']]
            for future in tqdm(as_completed(futures), total=len(futures), desc="Test"):
                test_data.append(future.result())

        # Shuffle training data
        random.shuffle(train_data)

        # Split train into train/val (LRA uses 25k train, but we create a val set)
        val_size = 5000
        val_data = train_data[:val_size]
        train_data = train_data[val_size:]

        return train_data, val_data, test_data

In [ ]:
gen = IMDbGenerator(max_len=4096, seed=42, device=device)
print(f"Using device: {gen.device}")
print(f"Vocab size: {gen.vocab_size} (byte-level)")
print(f"Max sequence length: {gen.max_len}\n")

# Load and process IMDb dataset
train_data, val_data, test_data = gen.load_and_process(num_workers=8)

# Create PyTorch datasets
train_ds = IMDbDataset(train_data, max_len=4096)
val_ds = IMDbDataset(val_data, max_len=4096)
test_ds = IMDbDataset(test_data, max_len=4096)

# Save
torch.save({
    'train': train_ds,
    'val': val_ds,
    'test': test_ds,
    'vocab_size': gen.vocab_size,
    'max_len': gen.max_len,
    'num_classes': 2
}, 'imdb_lra.pt')

print(f"\nSaved to imdb_lra.pt")
print(f"Vocab size: {gen.vocab_size}")
print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")


### 4.4 Retrieval: Document similarity matching (ACL Anthology)


Can the model tell if two documents are related by comparing their full contents?

Link: https://huggingface.co/datasets/WINGNUS/ACL-OCL

In [ ]:
class ACLRetrievalGenerator:
    """
    ACL Anthology document retrieval for Long Range Arena benchmark.
    - Binary classification: are two papers similar (cited) or not?
    - Byte-level tokenization (vocab size 257)
    - Max sequence length: 4000 per document (8000 total for pair)
    - Similarity defined by citation relationship
    """
    def __init__(self, max_len_per_doc=4000, seed=42, device=None):
        random.seed(seed)
        np.random.seed(seed)
        self.max_len_per_doc = max_len_per_doc
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        # Byte-level vocabulary (same as IMDb for consistency)
        self.vocab_size = 257  # 0 = PAD, 1-256 = byte values
        self.pad_token = 0
        self.sep_token = 256  # Special separator between documents

    def tokenize(self, text):
        """Convert text to byte-level tokens."""
        if text is None or (isinstance(text, float) and np.isnan(text)):
            return []
        text = str(text)
        return [b + 1 for b in text.encode('utf-8', errors='replace')]

    def load_dataset(self):
        """Load ACL-OCL dataset from HuggingFace."""
        print("Loading ACL-OCL dataset from HuggingFace...")
        print("Dataset: WINGNUS/ACL-OCL")

        import pandas as pd
        from huggingface_hub import hf_hub_download

        # Download the parquet file from HuggingFace Hub
        # Note: filename is "acl-publication-info.74k.v2.parquet" (with v2)
        print("Downloading paper metadata (515 MB)...")
        parquet_path = hf_hub_download(
            repo_id="WINGNUS/ACL-OCL",
            filename="acl-publication-info.74k.v2.parquet",
            repo_type="dataset"
        )

        # Also download citation graph for true similarity pairs
        print("Downloading citation graph (73 MB)...")
        citations_path = hf_hub_download(
            repo_id="WINGNUS/ACL-OCL",
            filename="acl_full_citations.parquet",
            repo_type="dataset"
        )

        # Load with pandas
        print("Loading parquet files...")
        df = pd.read_parquet(parquet_path)
        citations_df = pd.read_parquet(citations_path)

        print(f"Loaded {len(df)} papers")
        print(f"Loaded {len(citations_df)} citation relationships")

        return df, citations_df

    def build_citation_graph(self, df, citations_df):
        """
        Build citation relationships from the citation graph.
        Papers that cite each other are considered similar.
        """
        print("Building citation-based similarity pairs...")

        # Filter papers with valid abstracts and full text
        df_valid = df[
            (df['abstract'].notna()) &
            (df['abstract'].str.len() > 100) &
            (df['full_text'].notna()) &
            (df['full_text'].str.len() > 200)
        ].copy()

        print(f"Papers with valid text: {len(df_valid)}")

        # Create paper index mapping acl_id to index
        papers = df_valid[['acl_id', 'abstract', 'full_text', 'title']].reset_index(drop=True)
        paper_id_to_idx = {pid: idx for idx, pid in enumerate(papers['acl_id'].tolist())}

        # Build citation pairs from citations_df
        # The citation df should have citing and cited paper IDs
        print(f"Citation graph columns: {citations_df.columns.tolist()}")

        return papers, paper_id_to_idx, citations_df

    def create_pairs(self, papers, paper_id_to_idx, citations_df, n_pairs, positive_ratio=0.5):
        """
        Create document pairs for retrieval task.
        - Positive pairs: papers with citation relationship
        - Negative pairs: random paper pairs without citation relationship
        """
        print(f"Creating {n_pairs} document pairs...")

        pairs = []
        n_positive = int(n_pairs * positive_ratio)
        n_negative = n_pairs - n_positive

        papers_list = papers.to_dict('records')
        n_papers = len(papers_list)
        valid_acl_ids = set(papers['acl_id'].tolist())

        # Try to use citation relationships for positive pairs
        # First, let's see what the citation df looks like
        citation_pairs = set()
        try:
            # Common column names for citation data
            for col_pair in [('citing', 'cited'), ('source', 'target'),
                            ('paper_id', 'cited_paper_id'), ('from', 'to')]:
                if col_pair[0] in citations_df.columns and col_pair[1] in citations_df.columns:
                    for _, row in citations_df.iterrows():
                        citing = str(row[col_pair[0]])
                        cited = str(row[col_pair[1]])
                        if citing in valid_acl_ids and cited in valid_acl_ids:
                            citation_pairs.add((citing, cited))
                    break

            # If column detection failed, try first two columns
            if len(citation_pairs) == 0 and len(citations_df.columns) >= 2:
                col1, col2 = citations_df.columns[0], citations_df.columns[1]
                for _, row in citations_df.head(10000).iterrows():
                    citing = str(row[col1])
                    cited = str(row[col2])
                    if citing in valid_acl_ids and cited in valid_acl_ids:
                        citation_pairs.add((citing, cited))
        except Exception as e:
            print(f"Warning: Could not parse citations: {e}")

        print(f"Found {len(citation_pairs)} valid citation pairs")

        # Create positive pairs from citations (or fallback to proximity)
        print(f"  Creating {n_positive} positive pairs...")
        citation_list = list(citation_pairs)

        for i in tqdm(range(n_positive), desc="Positive pairs"):
            if citation_list and i < len(citation_list):
                # Use actual citation relationship
                citing_id, cited_id = citation_list[i % len(citation_list)]
                idx1 = paper_id_to_idx[citing_id]
                idx2 = paper_id_to_idx[cited_id]
            else:
                # Fallback: use nearby papers (same venue/time proxy)
                idx1 = random.randint(0, n_papers - 50)
                offset = random.randint(1, min(49, n_papers - idx1 - 1))
                idx2 = idx1 + offset

            doc1_text = str(papers_list[idx1].get('abstract', '')) + " " + str(papers_list[idx1].get('full_text', ''))[:2000]
            doc2_text = str(papers_list[idx2].get('abstract', '')) + " " + str(papers_list[idx2].get('full_text', ''))[:2000]

            pairs.append((doc1_text, doc2_text, 1))  # label 1 = similar

        # Create negative pairs (random dissimilar papers)
        print(f"  Creating {n_negative} negative pairs...")
        for _ in tqdm(range(n_negative), desc="Negative pairs"):
            idx1 = random.randint(0, n_papers - 1)
            # Pick a distant paper (at least 1000 papers apart)
            idx2 = (idx1 + random.randint(1000, n_papers - 1)) % n_papers

            doc1_text = str(papers_list[idx1].get('abstract', '')) + " " + str(papers_list[idx1].get('full_text', ''))[:2000]
            doc2_text = str(papers_list[idx2].get('abstract', '')) + " " + str(papers_list[idx2].get('full_text', ''))[:2000]

            pairs.append((doc1_text, doc2_text, 0))  # label 0 = dissimilar

        random.shuffle(pairs)
        return pairs

    def process_pairs(self, pairs):
        """Convert document pairs to tokenized format."""
        print("Tokenizing document pairs...")
        processed = []

        for doc1, doc2, label in tqdm(pairs, desc="Tokenizing"):
            tokens1 = self.tokenize(doc1)[:self.max_len_per_doc]
            tokens2 = self.tokenize(doc2)[:self.max_len_per_doc]

            # Concatenate: [doc1] [SEP] [doc2]
            combined = tokens1 + [self.sep_token] + tokens2
            processed.append((combined, label))

        return processed


In [ ]:
# Initialize generator
gen = ACLRetrievalGenerator(max_len_per_doc=4000, seed=42, device=device)
print(f"Using device: {gen.device}")
print(f"Vocab size: {gen.vocab_size} (byte-level)")
print(f"Max length per document: {gen.max_len_per_doc}")
print(f"Total max sequence length: {gen.max_len_per_doc * 2 + 1}\n")

# Load ACL-OCL dataset (both papers and citations)
df, citations_df = gen.load_dataset()

# Build paper index with citation graph
papers, paper_id_to_idx, citations_df = gen.build_citation_graph(df, citations_df)

# Create train/val/test splits
print("\n" + "="*50)
print("Creating dataset splits...")
print("="*50)

train_pairs = gen.create_pairs(papers, paper_id_to_idx, citations_df, n_pairs=20000)
val_pairs = gen.create_pairs(papers, paper_id_to_idx, citations_df, n_pairs=2000)
test_pairs = gen.create_pairs(papers, paper_id_to_idx, citations_df, n_pairs=2000)

# Process pairs
train_data = gen.process_pairs(train_pairs)
val_data = gen.process_pairs(val_pairs)
test_data = gen.process_pairs(test_pairs)

# Create PyTorch datasets
max_seq_len = gen.max_len_per_doc * 2 + 1  # doc1 + sep + doc2
train_ds = ACLRetrievalDataset(train_data, max_len=max_seq_len)
val_ds = ACLRetrievalDataset(val_data, max_len=max_seq_len)
test_ds = ACLRetrievalDataset(test_data, max_len=max_seq_len)

# Save
torch.save({
    'train': train_ds,
    'val': val_ds,
    'test': test_ds,
    'vocab_size': gen.vocab_size,
    'max_len': max_seq_len,
    'num_classes': 2,
    'task': 'retrieval'
}, 'acl_retrieval_lra.pt')

print(f"\n" + "="*50)
print("DATASET SAVED SUCCESSFULLY")
print("="*50)
print(f"Saved to: acl_retrieval_lra.pt")
print(f"Vocab size: {gen.vocab_size}")
print(f"Max sequence length: {max_seq_len}")
print(f"Number of classes: 2 (similar/dissimilar)")
print(f"\nSplit sizes:")
print(f"  Train: {len(train_ds):,} pairs")
print(f"  Val:   {len(val_ds):,} pairs")
print(f"  Test:  {len(test_ds):,} pairs")

# Verify a sample
print(f"\nSample verification:")
sample = train_ds[0]
print(f"  Input shape: {sample['input_ids'].shape}")
print(f"  Mask shape:  {sample['attention_mask'].shape}")
print(f"  Label:       {sample['labels'].item()}")

### 4.5 Image: Sequential CIFAR-10 classification (pixels as sequence)

Can the model recognize an image when pixels are fed one at a time in a long line?

In [ ]:
class CIFAR10SequentialGenerator:
    """
    CIFAR-10 sequential image classification for Long Range Arena benchmark.
    - 10-class classification
    - Images flattened to 1D sequence: 32x32x3 = 3072 tokens
    - Each pixel channel is a token (grayscale version: 32x32 = 1024 tokens)
    - Vocab size: 256 (pixel values 0-255)
    """
    def __init__(self, grayscale=True, seed=42, device=None):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)

        self.grayscale = grayscale
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        # Pixel vocabulary (0-255) + PAD token
        self.vocab_size = 257  # 0 = PAD, 1-256 = pixel values (shifted by 1)
        self.pad_token = 0

        # Sequence length depends on grayscale vs color
        # LRA uses grayscale: 32x32 = 1024
        # Color would be: 32x32x3 = 3072
        self.seq_len = 1024 if grayscale else 3072
        self.num_classes = 10

        # CIFAR-10 class names
        self.class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
                           'dog', 'frog', 'horse', 'ship', 'truck']

    def load_dataset(self):
        """Load CIFAR-10 dataset from HuggingFace."""
        print("Loading CIFAR-10 dataset from HuggingFace...")

        dataset = load_dataset('cifar10')

        print(f"Train samples: {len(dataset['train'])}")
        print(f"Test samples: {len(dataset['test'])}")

        return dataset

    def process_image(self, image):
        """
        Convert PIL image to sequential tokens.
        - Convert to grayscale if specified
        - Flatten to 1D sequence
        - Shift pixel values by 1 (so 0 can be PAD)
        """
        # Convert PIL to numpy
        img_array = np.array(image)

        if self.grayscale:
            # Convert RGB to grayscale: 0.299*R + 0.587*G + 0.114*B
            if len(img_array.shape) == 3:
                img_array = np.dot(img_array[..., :3], [0.299, 0.587, 0.114])
            img_array = img_array.astype(np.uint8)

        # Flatten to 1D sequence
        flat = img_array.flatten()

        # Shift by 1 so 0 is reserved for PAD
        tokens = [int(p) + 1 for p in flat]

        return tokens

    def process_dataset(self, dataset, split='train', num_workers=8):
        """Process entire split of CIFAR-10 to sequential format."""
        print(f"\nProcessing {split} split...")

        data = []
        examples = dataset[split]

        for i in tqdm(range(len(examples)), desc=f"Processing {split}"):
            image = examples[i]['img']
            label = examples[i]['label']

            tokens = self.process_image(image)
            data.append((tokens, label))

        return data

In [ ]:
gen = CIFAR10SequentialGenerator(grayscale=True, seed=42, device=device)
print(f"Using device: {gen.device}")
print(f"Vocab size: {gen.vocab_size} (pixel values)")
print(f"Sequence length: {gen.seq_len} ({'grayscale 32x32' if gen.grayscale else 'color 32x32x3'})")
print(f"Number of classes: {gen.num_classes}")
print(f"Classes: {gen.class_names}\n")

# Load CIFAR-10 dataset
dataset = gen.load_dataset()

# Process train and test sets
train_data = gen.process_dataset(dataset, split='train')
test_data = gen.process_dataset(dataset, split='test')

# Create validation split from training data (LRA uses 45k train, 5k val)
random.shuffle(train_data)
val_size = 5000
val_data = train_data[:val_size]
train_data = train_data[val_size:]

print(f"\nSplit sizes after validation split:")
print(f"  Train: {len(train_data)}")
print(f"  Val:   {len(val_data)}")
print(f"  Test:  {len(test_data)}")

# Create PyTorch datasets
train_ds = CIFAR10SequentialDataset(train_data, seq_len=gen.seq_len)
val_ds = CIFAR10SequentialDataset(val_data, seq_len=gen.seq_len)
test_ds = CIFAR10SequentialDataset(test_data, seq_len=gen.seq_len)

# Save
torch.save({
    'train': train_ds,
    'val': val_ds,
    'test': test_ds,
    'vocab_size': gen.vocab_size,
    'seq_len': gen.seq_len,
    'num_classes': gen.num_classes,
    'class_names': gen.class_names,
    'grayscale': gen.grayscale,
    'task': 'image_classification'
}, 'cifar10_sequential_lra.pt')

print(f"\n" + "="*50)
print("DATASET SAVED SUCCESSFULLY")
print("="*50)
print(f"Saved to: cifar10_sequential_lra.pt")
print(f"Vocab size: {gen.vocab_size}")
print(f"Sequence length: {gen.seq_len}")
print(f"Number of classes: {gen.num_classes}")
print(f"\nSplit sizes:")
print(f"  Train: {len(train_ds):,} images")
print(f"  Val:   {len(val_ds):,} images")
print(f"  Test:  {len(test_ds):,} images")

### 4.6 Two Pathfinder datasets

#### 4.6.1 Defining a common class for both

In [ ]:
class PathfinderGenerator:
    """
    GPU-accelerated Pathfinder generator for Long Range Arena benchmark.

    Task: Binary classification - are two dots connected by a path?
    - Pathfinder: 32x32 images -> 1024 sequence length
    - Path-X: 128x128 images -> 16384 sequence length
    """
    def __init__(self, resolution=32, seed=42, device=None,
                 contour_length=14, num_distractor_snakes=6, snake_contrast=0.5):
        self.seed = seed
        torch.manual_seed(seed)
        np.random.seed(seed)
        random.seed(seed)

        self.resolution = resolution
        self.seq_len = resolution * resolution
        self.device = device or ('cuda' if torch.cuda.is_available() else 'cpu')

        self.contour_length = contour_length
        self.num_distractor_snakes = num_distractor_snakes
        self.snake_contrast = snake_contrast

        self.vocab_size = 257
        self.pad_token = 0
        self.num_classes = 2

    def _draw_snake_batch(self, imgs, start_positions, lengths, mark_endpoints=False):
        """Draw snakes on a batch of images using GPU operations."""
        batch_size = imgs.shape[0]
        h, w = self.resolution, self.resolution
        device = imgs.device

        x = start_positions[:, 0].float()
        y = start_positions[:, 1].float()
        angles = torch.rand(batch_size, device=device) * 2 * np.pi

        start_x = x.clone()
        start_y = y.clone()

        max_length = lengths.max().item()

        for step in range(max_length):
            angles += torch.randn(batch_size, device=device) * 0.5
            dx = torch.cos(angles) * 2
            dy = torch.sin(angles) * 2
            x = torch.clamp(x + dx, 2, w - 3)
            y = torch.clamp(y + dy, 2, h - 3)
            active = step < lengths

            for ox in range(-1, 2):
                for oy in range(-1, 2):
                    px = (x + ox).long().clamp(0, w - 1)
                    py = (y + oy).long().clamp(0, h - 1)
                    batch_idx = torch.arange(batch_size, device=device)
                    intensity = (128 * self.snake_contrast * active.float()).long()
                    imgs[batch_idx, py, px] = torch.clamp(
                        imgs[batch_idx, py, px] + intensity, 0, 255
                    )

        if mark_endpoints:
            for pos_x, pos_y in [(start_x, start_y), (x, y)]:
                for ox in range(-2, 3):
                    for oy in range(-2, 3):
                        if ox*ox + oy*oy <= 4:
                            px = (pos_x + ox).long().clamp(0, w - 1)
                            py = (pos_y + oy).long().clamp(0, h - 1)
                            batch_idx = torch.arange(batch_size, device=device)
                            imgs[batch_idx, py, px] = 255

        return x.long(), y.long()

    def _generate_batch(self, batch_size):
        """Generate a batch of pathfinder samples on GPU."""
        h, w = self.resolution, self.resolution
        device = self.device
        margin = max(5, self.resolution // 8)

        imgs = torch.randint(0, 50, (batch_size, h, w), device=device, dtype=torch.long)
        labels = torch.randint(0, 2, (batch_size,), device=device)

        start_x = torch.randint(margin, w - margin, (batch_size,), device=device)
        start_y = torch.randint(margin, h - margin, (batch_size,), device=device)
        start_positions = torch.stack([start_x, start_y], dim=1)

        lengths = torch.full((batch_size,), self.contour_length, device=device)
        end_x, end_y = self._draw_snake_batch(imgs, start_positions, lengths, mark_endpoints=True)

        for _ in range(self.num_distractor_snakes):
            dist_x = torch.randint(margin, w - margin, (batch_size,), device=device)
            dist_y = torch.randint(margin, h - margin, (batch_size,), device=device)
            dist_positions = torch.stack([dist_x, dist_y], dim=1)
            dist_lengths = torch.randint(
                self.contour_length // 2,
                self.contour_length + 1,
                (batch_size,),
                device=device
            )
            self._draw_snake_batch(imgs, dist_positions, dist_lengths, mark_endpoints=False)

        not_connected = labels == 0
        if not_connected.any():
            new_x = torch.randint(margin, w - margin, (batch_size,), device=device)
            new_y = torch.randint(margin, h - margin, (batch_size,), device=device)

            for _ in range(10):
                too_close = (torch.abs(new_x - start_x) < w // 3) & (torch.abs(new_y - start_y) < h // 3)
                if not too_close.any():
                    break
                new_x = torch.where(too_close, torch.randint(margin, w - margin, (batch_size,), device=device), new_x)
                new_y = torch.where(too_close, torch.randint(margin, h - margin, (batch_size,), device=device), new_y)

            for ox in range(-2, 3):
                for oy in range(-2, 3):
                    if ox*ox + oy*oy <= 4:
                        px = (new_x + ox).clamp(0, w - 1)
                        py = (new_y + oy).clamp(0, h - 1)
                        batch_idx = torch.arange(batch_size, device=device)
                        imgs[batch_idx[not_connected], py[not_connected], px[not_connected]] = 255

        # Flatten and shift by 1 for PAD token
        tokens = imgs.view(batch_size, -1) + 1

        return tokens, labels

    def generate(self, n_samples, batch_size=1024):
        """Generate n_samples using GPU - returns tensors directly (no slow conversion)."""
        print(f"Generating {n_samples} samples on {self.device} (resolution={self.resolution}x{self.resolution})...")

        all_tokens = []
        all_labels = []

        num_batches = (n_samples + batch_size - 1) // batch_size

        for i in tqdm(range(num_batches), desc="Generating batches"):
            current_batch_size = min(batch_size, n_samples - i * batch_size)
            tokens, labels = self._generate_batch(current_batch_size)
            # Move to CPU immediately to free GPU memory
            all_tokens.append(tokens.cpu())
            all_labels.append(labels.cpu())

        # Single concatenation (fast)
        all_tokens = torch.cat(all_tokens, dim=0)
        all_labels = torch.cat(all_labels, dim=0)

        return all_tokens, all_labels

#### 4.6.2 Pathfinder: Visual path detection in synthetic images

Can the model trace a curvy line through noise to see if two dots are connected?

In [ ]:
gen_pathfinder = PathfinderGenerator(
    resolution=32,
    seed=42,
    device=device,
    contour_length=14,
    num_distractor_snakes=6
)

print(f"Using device: {gen_pathfinder.device}")
print(f"Resolution: {gen_pathfinder.resolution}x{gen_pathfinder.resolution}")
print(f"Sequence length: {gen_pathfinder.seq_len}")
print(f"Vocab size: {gen_pathfinder.vocab_size}")
print(f"Classes: 2 (connected / not connected)\n")

batch_size_pf = 2048 if torch.cuda.is_available() else 512

# Generate returns tensors directly now (no slow list conversion)
train_tokens_pf, train_labels_pf = gen_pathfinder.generate(160000, batch_size=batch_size_pf)
val_tokens_pf, val_labels_pf = gen_pathfinder.generate(20000, batch_size=batch_size_pf)
test_tokens_pf, test_labels_pf = gen_pathfinder.generate(20000, batch_size=batch_size_pf)

# Create PyTorch datasets (pass tensors directly)
train_ds_pf = PathfinderDataset(train_tokens_pf, train_labels_pf, seq_len=gen_pathfinder.seq_len)
val_ds_pf = PathfinderDataset(val_tokens_pf, val_labels_pf, seq_len=gen_pathfinder.seq_len)
test_ds_pf = PathfinderDataset(test_tokens_pf, test_labels_pf, seq_len=gen_pathfinder.seq_len)

# Save - now saves tensors directly (much faster!)
print("Saving dataset...")
torch.save({
    'train': train_ds_pf,
    'val': val_ds_pf,
    'test': test_ds_pf,
    'vocab_size': gen_pathfinder.vocab_size,
    'seq_len': gen_pathfinder.seq_len,
    'resolution': gen_pathfinder.resolution,
    'num_classes': gen_pathfinder.num_classes,
    'task': 'pathfinder'
}, 'pathfinder_lra.pt')

print(f"\n" + "-"*40)
print("PATHFINDER SAVED")
print("-"*40)
print(f"Saved to: pathfinder_lra.pt")
print(f"Train: {len(train_ds_pf):,} | Val: {len(val_ds_pf):,} | Test: {len(test_ds_pf):,}")

#### 4.6.3 Path-X: Extended version of Pathfinder with 16K sequence length

Same as Pathfinder but much harder because the sequence is 16 times longer.

In [ ]:
gen_pathx = PathfinderGenerator(
    resolution=128,
    seed=42,
    device=device,
    contour_length=48,
    num_distractor_snakes=12
)

print(f"Using device: {gen_pathx.device}")
print(f"Resolution: {gen_pathx.resolution}x{gen_pathx.resolution}")
print(f"Sequence length: {gen_pathx.seq_len}")
print(f"Vocab size: {gen_pathx.vocab_size}")
print(f"Classes: 2 (connected / not connected)\n")

batch_size_px = 512 if torch.cuda.is_available() else 128

# Generate returns tensors directly now (no slow list conversion)
train_tokens_px, train_labels_px = gen_pathx.generate(160000, batch_size=batch_size_px)
val_tokens_px, val_labels_px = gen_pathx.generate(20000, batch_size=batch_size_px)
test_tokens_px, test_labels_px = gen_pathx.generate(20000, batch_size=batch_size_px)

# Create PyTorch datasets (pass tensors directly)
train_ds_px = PathfinderDataset(train_tokens_px, train_labels_px, seq_len=gen_pathx.seq_len)
val_ds_px = PathfinderDataset(val_tokens_px, val_labels_px, seq_len=gen_pathx.seq_len)
test_ds_px = PathfinderDataset(test_tokens_px, test_labels_px, seq_len=gen_pathx.seq_len)

# Save - tensors save MUCH faster than Python lists
print("Saving dataset...")
torch.save({
    'train': train_ds_px,
    'val': val_ds_px,
    'test': test_ds_px,
    'vocab_size': gen_pathx.vocab_size,
    'seq_len': gen_pathx.seq_len,
    'resolution': gen_pathx.resolution,
    'num_classes': gen_pathx.num_classes,
    'task': 'pathx'
}, 'pathx_lra.pt')

print(f"\n" + "-"*40)
print("PATH-X SAVED")
print("-"*40)
print(f"Saved to: pathx_lra.pt")
print(f"Train: {len(train_ds_px):,} | Val: {len(val_ds_px):,} | Test: {len(test_ds_px):,}")

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"\nGPU memory cleared")

### 4.7 Sending data to hugging face

In [ ]:
login()

api = HfApi()
repo_id = "monteirot/lra-benchmarks"

# Create the repo
api.create_repo(repo_id, repo_type="dataset", exist_ok=True)

# Upload all saved .pt files
for f in ['listops.pt', 'imdb_lra.pt', 'acl_retrieval_lra.pt',
          'cifar10_sequential_lra.pt', 'pathfinder_lra.pt', 'pathx_lra.pt']:
    print(f"Uploading {f}...")
    api.upload_file(path_or_fileobj=f, path_in_repo=f,
                    repo_id=repo_id, repo_type="dataset")
    print(f"  ✓ {f} uploaded")

print(f"\nDone! Dataset at: https://huggingface.co/datasets/{repo_id}")

HfHubHTTPError: Client error '401 Unauthorized' for url 'https://huggingface.co/api/repos/create' (Request ID: Root=1-698ea6e1-3bc2b7ca3030d9eb033279e1;6d2e6608-f9b3-4c40-9591-5106320d8a55)
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/401

Invalid username or password.